In [11]:
import requests
import pandas as pd

API_BASE_URL = "http://127.0.0.1:8000"
RESUME_ID = 8

response = requests.get(
    f"{API_BASE_URL}/jobs",
    params={
        "limit": 50,
        "offset": 0,
        "source_filter": "all",
        "freshness": "all",
        "relevant_only": "false",
        "resume_id": RESUME_ID,
    }
)

print("Status:", response.status_code)
print("URL:", response.url)
print("Response preview:")
print(response.text[:1000])

Status: 200
URL: http://127.0.0.1:8000/jobs?limit=50&offset=0&source_filter=all&freshness=all&relevant_only=false&resume_id=8
Response preview:
[{"id":1714,"source":"indeed_apify","source_job_id":"15911b4cc1cc5816","title":"Lead Software Engineer – Java Data Engineering","company":"Strategic Staffing Solutions","location":"Plano, TX 75024","description":"Lead Software Engineer – Java Data Engineering *Location:* Plano, TX (Onsite – 4 days/week) *Duration:* Contract *Industry:* Financial Services Position Overview Seeking a Lead Software Engineer with strong Java and Data Engineering expertise to design, build, and maintain scalable enterprise data pipelines and backend processing solutions. This role will focus on ingesting, validating, transforming, and integrating large volumes of data while supporting enterprise identity, access governance, and cybersecurity initiatives. The ideal candidate has extensive experience developing Java-based backend applications, building high-performanc

In [12]:
response.raise_for_status()
jobs = response.json()

jobs_df = pd.DataFrame([
    {
        "job_id": job.get("id"),
        "title": job.get("title"),
        "company": job.get("company"),
        "location": job.get("location"),
        "ats_score": job.get("ats_score"),
        "ml_preference_score": job.get("ml_preference_score"),
        "recommended_score": job.get("recommended_score"),
    }
    for job in jobs
])

print("Jobs loaded:", len(jobs_df))
jobs_df.head(20)

Jobs loaded: 50


,job_id,title,company,location,ats_score,ml_preference_score,recommended_score
0,1714,Lead Software Engineer – Java Data Engineering,Strategic Staffing Solutions,"Plano, TX 75024",85,93.76,88
1,1711,ETL Focused Data Engineer,Experis,"Charlotte, NC 28202",82,91.13,85
2,1671,Senior Business Intelligence Developer-Knoxvil...,Edfinancial Services LLC,"Knoxville, TN 37922",80,95.82,85
3,1674,Senior Data Architect,System One,"Washington, DC 20006",79,93.24,83
4,1701,Software Engineer III - Python Automation,JPMorganChase,"Jersey City, NJ 07310",85,55.02,76
5,1679,Software Developer,FTI Defense - Frontier Technology Inc.,"Norfolk, VA",83,59.69,76
6,1700,Senior Software Engineer Mission Systems Softw...,Boeing,"Daytona Beach, FL 32198",81,59.69,75
7,1685,AI Infra Engineer,Harrison Clarke,San Francisco Bay Area,83,56.83,75
8,1683,Backend Software Developer - PERM Direct Hire,Robert Half,"Des Moines, IA",82,56.83,74
9,1675,Senior Data Engineer,ienjoy Home,"Clearwater, FL 33765",79,63.87,74


In [13]:
# Use ATS score first. If ATS score is missing, use recommended score.
jobs_for_feedback = jobs_df.copy()

jobs_for_feedback["score_for_selection"] = jobs_for_feedback["ats_score"]

if jobs_for_feedback["score_for_selection"].isna().all():
    jobs_for_feedback["score_for_selection"] = jobs_for_feedback["recommended_score"]

jobs_for_feedback = jobs_for_feedback.dropna(subset=["job_id", "score_for_selection"]).copy()

jobs_for_feedback["job_id"] = jobs_for_feedback["job_id"].astype(int)

positive_job_ids = (
    jobs_for_feedback
    .sort_values("score_for_selection", ascending=False)
    .head(5)["job_id"]
    .tolist()
)

negative_job_ids = (
    jobs_for_feedback
    .sort_values("score_for_selection", ascending=True)
    .head(5)["job_id"]
    .tolist()
)

print("Positive job IDs:", positive_job_ids)
print("Negative job IDs:", negative_job_ids)

Positive job IDs: [1714, 1701, 1685, 1679, 1711]
Negative job IDs: [1694, 1666, 1702, 1684, 1680]


In [14]:
created = []
failed = []

for job_id in positive_job_ids:
    payload = {
        "resume_id": RESUME_ID,
        "job_id": int(job_id),
        "feedback_label": "interested",
    }

    response = requests.post(f"{API_BASE_URL}/feedback", json=payload)

    print("Positive:", job_id, response.status_code, response.text[:200])

    if response.status_code in [200, 201]:
        created.append(response.json())
    else:
        failed.append((job_id, response.status_code, response.text[:300]))

for job_id in negative_job_ids:
    payload = {
        "resume_id": RESUME_ID,
        "job_id": int(job_id),
        "feedback_label": "not_interested",
    }

    response = requests.post(f"{API_BASE_URL}/feedback", json=payload)

    print("Negative:", job_id, response.status_code, response.text[:200])

    if response.status_code in [200, 201]:
        created.append(response.json())
    else:
        failed.append((job_id, response.status_code, response.text[:300]))

print("Created feedback records:", len(created))
print("Failed records:", failed)

Positive: 1714 200 {"feedback_id":86,"message":"Feedback saved successfully."}
Positive: 1701 200 {"feedback_id":87,"message":"Feedback saved successfully."}
Positive: 1685 200 {"feedback_id":88,"message":"Feedback saved successfully."}
Positive: 1679 200 {"feedback_id":89,"message":"Feedback saved successfully."}
Positive: 1711 200 {"feedback_id":90,"message":"Feedback saved successfully."}
Negative: 1694 200 {"feedback_id":91,"message":"Feedback saved successfully."}
Negative: 1666 200 {"feedback_id":92,"message":"Feedback saved successfully."}
Negative: 1702 200 {"feedback_id":93,"message":"Feedback saved successfully."}
Negative: 1684 200 {"feedback_id":94,"message":"Feedback saved successfully."}
Negative: 1680 200 {"feedback_id":95,"message":"Feedback saved successfully."}
Created feedback records: 10
Failed records: []


In [15]:
response = requests.get(f"{API_BASE_URL}/feedback", params={"limit": 1000})
response.raise_for_status()

feedback_df = pd.DataFrame(response.json())

print("Total feedback records:", len(feedback_df))
feedback_df.head(20)

Total feedback records: 36


,feedback_id,resume_id,job_id,feedback_label,created_at,title,company,location,apply_url
0,95,8,1680,not_interested,2026-06-29 23:48:06.316951,Software Engineer II - Fullstack - Python/React,JPMorganChase,"New York, NY",https://www.linkedin.com/jobs/view/software-en...
1,94,8,1684,not_interested,2026-06-29 23:48:06.301605,"Software Engineer, Generative AI",Jobright.ai,"New York, United States",https://www.linkedin.com/jobs/view/software-en...
2,93,8,1702,not_interested,2026-06-29 23:48:06.283290,Software Engineer II - Infrastructure,Disney Entertainment and ESPN Product & Techno...,"New York, NY 10013",https://www.indeed.com/viewjob?jk=57fc46af20c4...
3,92,8,1666,not_interested,2026-06-29 23:48:06.266465,"Product Design Electrical Engineer, Data Cente...","Amazon Data Services, Inc.","Seattle, WA",https://www.indeed.com/viewjob?jk=6c1c4d4fa4d4...
4,91,8,1694,not_interested,2026-06-29 23:48:06.249463,Python Developer,BeaconFire Inc.,"New Jersey, United States",https://www.linkedin.com/jobs/view/python-deve...
5,90,8,1711,interested,2026-06-29 23:48:06.232395,ETL Focused Data Engineer,Experis,"Charlotte, NC 28202",https://www.indeed.com/viewjob?jk=0a0f77391612...
6,89,8,1679,interested,2026-06-29 23:48:06.215444,Software Developer,FTI Defense - Frontier Technology Inc.,"Norfolk, VA",https://www.linkedin.com/jobs/view/software-de...
7,88,8,1685,interested,2026-06-29 23:48:06.198277,AI Infra Engineer,Harrison Clarke,San Francisco Bay Area,https://www.linkedin.com/jobs/view/ai-infra-en...
8,87,8,1701,interested,2026-06-29 23:48:06.183984,Software Engineer III - Python Automation,JPMorganChase,"Jersey City, NJ 07310",https://www.indeed.com/viewjob?jk=3686ce47d536...
9,86,8,1714,interested,2026-06-29 23:48:06.160472,Lead Software Engineer – Java Data Engineering,Strategic Staffing Solutions,"Plano, TX 75024",https://www.indeed.com/viewjob?jk=15911b4cc1cc...


In [16]:
response = requests.post(f"{API_BASE_URL}/ml/train-xgb-ranker")
response.raise_for_status()

training_result = response.json()
training_result

{'trained': True,
 'training_rows': 36,
 'positive_labels': 17,
 'negative_labels': 19,
 'model_path': 'backend\\models\\xgb_job_ranker.json',
 'feature_columns': ['ats_compatibility_score',
  'baseline_final_score',
  'semantic_score',
  'skill_score',
  'required_skill_score',
  'preferred_skill_score',
  'experience_score',
  'title_score',
  'education_score',
  'keyword_context_score',
  'location_score',
  'knockout_count',
  'matched_skills_count',
  'missing_skills_count',
  'job_skills_count',
  'missing_skill_ratio',
  'source_is_linkedin',
  'source_is_indeed',
  'title_has_data',
  'title_has_engineer',
  'title_has_analyst',
  'title_has_python',
  'title_has_software',
  'text_has_sql',
  'text_has_python',
  'text_has_etl',
  'text_has_cloud'],
 'feature_importance': [{'feature': 'baseline_final_score',
   'importance': 0.3961537480354309},
  {'feature': 'ats_compatibility_score', 'importance': 0.18618644773960114},
  {'feature': 'semantic_score', 'importance': 0.1336432

In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Reload scored jobs after training
response = requests.get(
    f"{API_BASE_URL}/jobs",
    params={
        "limit": 1000,
        "offset": 0,
        "source_filter": "all",
        "freshness": "all",
        "relevant_only": "false",
        "resume_id": RESUME_ID,
    }
)

response.raise_for_status()
jobs_after_training = response.json()

jobs_eval_df = pd.DataFrame([
    {
        "job_id": job.get("id"),
        "title": job.get("title"),
        "company": job.get("company"),
        "ats_score": job.get("ats_score"),
        "ml_preference_score": job.get("ml_preference_score"),
        "recommended_score": job.get("recommended_score"),
    }
    for job in jobs_after_training
])

# Reload feedback
response = requests.get(f"{API_BASE_URL}/feedback", params={"limit": 1000})
response.raise_for_status()
feedback_df = pd.DataFrame(response.json())

def label_to_binary(label):
    label = str(label).strip().lower().replace(" ", "_")

    if label in ["interested", "applied", "saved"]:
        return 1

    if label in ["not_interested", "rejected"]:
        return 0

    return None

feedback_df["y_true"] = feedback_df["feedback_label"].apply(label_to_binary)
feedback_df = feedback_df.dropna(subset=["y_true"]).copy()
feedback_df["y_true"] = feedback_df["y_true"].astype(int)

eval_df = feedback_df.merge(
    jobs_eval_df,
    on="job_id",
    how="inner"
)

# Prefer ML preference score. If missing, use recommended score.
if "ml_preference_score" in eval_df.columns and not eval_df["ml_preference_score"].isna().all():
    score_col = "ml_preference_score"
else:
    score_col = "recommended_score"

eval_df = eval_df.dropna(subset=[score_col]).copy()

threshold = 50
eval_df["y_pred"] = (eval_df[score_col] >= threshold).astype(int)

y_true = eval_df["y_true"]
y_pred = eval_df["y_pred"]

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [accuracy, precision, recall, f1],
    "Percentage": [
        round(accuracy * 100, 2),
        round(precision * 100, 2),
        round(recall * 100, 2),
        round(f1 * 100, 2),
    ]
})

metrics_df

,Metric,Score,Percentage
0,Accuracy,0.944444,94.44
1,Precision,0.894737,89.47
2,Recall,1.000000,100.00
3,F1 Score,0.944444,94.44


In [18]:
cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)

cm_df

,Predicted Negative,Predicted Positive
Actual Negative,17,2
Actual Positive,0,17


In [19]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Negative Feedback", "Positive Feedback"],
        zero_division=0
    )
)

                   precision    recall  f1-score   support

Negative Feedback       1.00      0.89      0.94        19
Positive Feedback       0.89      1.00      0.94        17

         accuracy                           0.94        36
        macro avg       0.95      0.95      0.94        36
     weighted avg       0.95      0.94      0.94        36

